# Step 1: Import Datasets

In [ ]:
%pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

# Step 2: Tokenize the dataset
1. Tokenize the dataset into tokenIDs.
2. Create a file called "train.bin" and "validation.bin" where we will store the tokenIDs from the entire dataset
3. We make sure the tokenIDs are stored on a disk rather than on the RAM for efficient computation

In [ ]:
%pip install -q tiktoken

In [10]:
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

In [11]:
enc

<Encoding 'gpt2'>

In [12]:
## utility fuction for converting text into token ids
def process(example):
    ids = enc.encode_ordinary(example['text'])
    output = {
        "ids": ids,
        "len": len(ids)
    }
    return output

In [15]:
process(ds['train'][0])

{'ids': [3198,
  1110,
  11,
  257,
  1310,
  2576,
  3706,
  20037,
  1043,
  257,
  17598,
  287,
  607,
  2119,
  13,
  1375,
  2993,
  340,
  373,
  2408,
  284,
  711,
  351,
  340,
  780,
  340,
  373,
  7786,
  13,
  20037,
  2227,
  284,
  2648,
  262,
  17598,
  351,
  607,
  1995,
  11,
  523,
  673,
  714,
  34249,
  257,
  4936,
  319,
  607,
  10147,
  13,
  198,
  198,
  43,
  813,
  1816,
  284,
  607,
  1995,
  290,
  531,
  11,
  366,
  29252,
  11,
  314,
  1043,
  428,
  17598,
  13,
  1680,
  345,
  2648,
  340,
  351,
  502,
  290,
  34249,
  616,
  10147,
  1701,
  2332,
  1995,
  13541,
  290,
  531,
  11,
  366,
  5297,
  11,
  20037,
  11,
  356,
  460,
  2648,
  262,
  17598,
  290,
  4259,
  534,
  10147,
  526,
  198,
  198,
  41631,
  11,
  484,
  4888,
  262,
  17598,
  290,
  384,
  19103,
  262,
  4936,
  319,
  20037,
  338,
  10147,
  13,
  632,
  373,
  407,
  2408,
  329,
  606,
  780,
  484,
  547,
  7373,
  290,
  5742,
  1123,
  584,
  13,
  2293,

In [19]:
os.path.exists("train.bin")

False

In [ ]:
!nproc ## number of cpu cores

2


In [21]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4399.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [33]:
if not os.path.exists("train.bin"):
    # if train.bin does exists then 
    tokenization = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the text/splits",
        num_proc=2
    )
    for split, dataset in tokenization.items():
        arr_len = np.sum(dataset['len'], dtype=np.uint64) ## sum total of len column with datatype of np.uint64
        filename = f"{split}.bin"
        dtype = np.uint16 ## we can use vocab size of gpt2 = 50256, < 2**16
        arr = np.memmap( ## cretaes disk based array
            filename,   
            dtype=dtype, ## data type of array 
            mode="w+", ## crete or overwrite the content
            shape=(arr_len,) ## shape of array here it's 1-D
        )

        total_batch = 1024 ## let's process the data in the batch of 1024
        idx = 0 ## current write position in memmap
        for batch_idx in tqdm(range(total_batch), desc=f"writting {filename}"):
            """
            iter1 = 0 - 1023, idx = 0+len(arr_batch)
            iter2 = 
            """
            batch = dataset.shard(
                num_shards=total_batch,
                index=batch_idx,
                contiguous=True,
            ).with_format("numpy")
        
            arr_batch = np.concatenate(batch['ids'])
            arr[idx: idx+len(arr_batch)] = arr_batch
            idx += len(arr_batch)
    arr.flush()

In [29]:
np.sum(tokenization['train'].to_pandas()['len'])

np.int64(471872517)

In [34]:
!ls

sample_data  train.bin	validation.bin


In [ ]:
!cat validation.bin 

 :�bTV� � ��V �R I>�"`
 _^|"xM��
��V"C
 s�[u�  _">Lv�1Lv�1�i)� � 1 �[>"Lv�1"���
FV? s�[">#��	 "��M"
 <"��C	<��1��M�uz[0�4
 b	� du�X�&�	<
 _k�&u�3�	_k�&�jn>�ex�F��	� n� �3M�� � 2 ]��h"�T?
 b	� du�/X��G��
 � � 1� �p��
 wcSf ZpBTG1T T��
 _&��p ���j	
 ��_�5��z>
 ���5>�	��?H�p
 � � 5f�_5
 ��5
 ��"� �5
  ��/5 �� z � I
 5y�
 �" �  >u�
 ��_�5uzZ
 YYlV Z2	�_�5
 � � �5�/5
 5n

 
 ZukS��(�5
 � � �� Z�5
 Z�S/��5
 ��S ��5
 Z
 � S��  �"<C
��	 "�
 b	� du
 ���hjX �d�)T�nd�� � \	 �}"Z
 w5"8�5T"s�
 �&� �*5uz�"� �}
 ��5
 YV �r
 �` �eR